# S³ Spectral-Spatial Domain-Adaptive EEG Research Notebook
This notebook implements the supplied architecture with research logging and paper-ready figures.

**Important:** the supplied `SimplifiedBiMamba` class is implemented with a bidirectional GRU, not a true Mamba SSM; the latent diffusion module described in the reference manuscript is also not part of the pasted implementation.

In [ ]:

# ============================================================
# S3 Spectral-Spatial Domain Adaptation EEG Research Pipeline
# Code-aligned with the supplied implementation
#
# IMPORTANT IMPLEMENTATION NOTE
# ------------------------------------------------------------
# The supplied model contains:
#   1) per-trial channel-wise Z-score normalization
#   2) learnable Sinc filter bank
#   3) dynamic graph neural network (DGNN)
#   4) "SimplifiedBiMamba" class implemented with a bidirectional GRU
#   5) squeeze-and-excitation attention
#   6) class head + GRL domain head + supervised contrastive head
#   7) AdaBN test-time adaptation
#
# It does NOT implement the latent diffusion module described in the
# attached reference manuscript, and its "Mamba" block is currently
# a bidirectional GRU approximation. The paper should therefore not
# claim true Mamba or diffusion results unless those modules are
# implemented separately.
# ============================================================

from pathlib import Path
import os, math, json, random, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mne
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    accuracy_score, cohen_kappa_score, confusion_matrix,
    classification_report, roc_curve, auc
)
from sklearn.manifold import TSNE

warnings.filterwarnings("ignore")
mne.set_log_level("ERROR")

# -----------------------------
# 0. CONFIGURATION
# -----------------------------
SEED = 42
DATA_DIR = "./eegmmidb"

TOTAL_SUBJECTS = 109
NUM_TEST_FOLDS = 10
NUM_TRAIN_SUBJECTS = 99

RUNS = [4, 6, 8, 10, 12, 14]
TMIN = 0.0
TMAX = 4.0
FS = 250.0
N_CHANNELS = 22
N_CLASSES = 4

BATCH_SIZE = 64
NUM_EPOCHS = 100
LR = 1e-3
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.1
DOMAIN_WEIGHT = 1.0
SUPCON_WEIGHT = 0.5
SUPCON_TEMP = 0.07
GRAD_CLIP = 1.0

NUM_FILTERS = 10
SINC_KERNEL = 81
SPATIAL_DIM = 64
DOMAIN_CLASSES = TOTAL_SUBJECTS

RESULTS_DIR = Path("./results_s3_da")
FIG_DIR = RESULTS_DIR / "figures"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = [
    "Left Fist",
    "Right Fist",
    "Both Fists",
    "Both Feet",
]

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything()

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print("Device:", DEVICE)
print("Torch:", torch.__version__)


# -----------------------------
# 1. DATASET
# -----------------------------
class EEGMMIDB_Dataset(Dataset):
    """
    Loads the same subject/run selection logic as the supplied code.

    EEGMMIDB mapping:
      Runs 4,8,12 -> T1/T2 = left/right fist imagery
      Runs 6,10,14 -> T1/T2 = both-fists/both-feet imagery
    The run-dependent mapping is important for the PhysioNet dataset.
    """
    def __init__(
        self,
        data_dir,
        subjects,
        runs=RUNS,
        tmin=TMIN,
        tmax=TMAX
    ):
        self.data_dir = str(data_dir)
        self.subjects = list(subjects)
        self.runs = list(runs)
        self.tmin = tmin
        self.tmax = tmax

        self.epochs = []
        self.labels = []
        self.subject_ids = []
        self.run_ids = []

        self.load_data()

    def load_data(self):
        for sub in self.subjects:
            sub_folder = f"S{sub:03d}"
            sub_path = os.path.join(self.data_dir, sub_folder)
            if not os.path.exists(sub_path):
                continue

            for run in self.runs:
                edf_file = os.path.join(sub_path, f"{sub_folder}R{run:02d}.edf")
                if not os.path.exists(edf_file):
                    continue

                try:
                    raw = mne.io.read_raw_edf(edf_file, preload=True, verbose=False)

                    # Match supplied implementation: first 22 channels.
                    raw.pick(raw.ch_names[:N_CHANNELS])
                    raw.resample(FS, npad="auto")

                    events, _ = mne.events_from_annotations(raw, verbose=False)

                    mapping = {}
                    if run in [4, 8, 12]:
                        mapping = {"T1": 0, "T2": 1}
                    elif run in [6, 10, 14]:
                        mapping = {"T1": 2, "T2": 3}

                    if not mapping:
                        continue

                    ep = mne.Epochs(
                        raw,
                        events,
                        event_id=mapping,
                        tmin=self.tmin,
                        tmax=self.tmax - 1.0 / FS,
                        baseline=None,
                        preload=True,
                        verbose=False
                    )

                    data = ep.get_data()
                    labels = ep.events[:, -1]

                    for i in range(len(data)):
                        self.epochs.append(data[i].astype(np.float32))
                        self.labels.append(int(labels[i]))
                        self.subject_ids.append(int(sub - 1))  # zero-based domain class
                        self.run_ids.append(int(run))

                except Exception as e:
                    # Keep the pipeline robust to individual EDF failures.
                    print(f"[WARN] S{sub:03d} R{run:02d}: {type(e).__name__}: {e}")

    def __len__(self):
        return len(self.epochs)

    def __getitem__(self, idx):
        x = torch.tensor(self.epochs[idx], dtype=torch.float32)
        y = torch.tensor(self.labels[idx], dtype=torch.long)
        s = torch.tensor(self.subject_ids[idx], dtype=torch.long)

        # Per-trial, per-channel robust Z-score as implemented in the supplied code.
        mean = x.mean(dim=1, keepdim=True)
        std = x.std(dim=1, keepdim=True)
        x = (x - mean) / (std + 1e-6)

        return x, y, s


def discover_available_subjects(data_dir, max_subjects=TOTAL_SUBJECTS):
    available = []
    for s in range(1, max_subjects + 1):
        if os.path.isdir(Path(data_dir) / f"S{s:03d}"):
            available.append(s)
    return available


# -----------------------------
# 2. ARCHITECTURE
# -----------------------------
class GradientReversalLayer(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambda_grl):
        ctx.lambda_grl = lambda_grl
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return grad_output.neg() * ctx.lambda_grl, None

def grl(x, lambda_grl=1.0):
    return GradientReversalLayer.apply(x, lambda_grl)


class SincFilterBank(nn.Module):
    def __init__(self, in_channels=22, num_filters=10, kernel_size=81, sample_rate=250):
        super().__init__()
        self.num_filters = num_filters
        self.kernel_size = kernel_size
        self.sample_rate = sample_rate

        # Same initialization family as the supplied code.
        self.f1 = nn.Parameter(torch.rand(num_filters) * 10 + 5)
        self.f2 = nn.Parameter(torch.rand(num_filters) * 20 + 15)

    def forward(self, x):
        B, C, T = x.shape

        n = torch.arange(
            -(self.kernel_size // 2),
            (self.kernel_size // 2) + 1,
            device=x.device,
            dtype=x.dtype
        )

        filters = []
        for i in range(self.num_filters):
            # Sorted positive cutoffs make the implementation numerically safer
            # without changing the conceptual learnable-Sinc design.
            lo = torch.minimum(self.f1[i], self.f2[i] - 1e-3).clamp(0.5, 70.0)
            hi = torch.maximum(self.f2[i], self.f1[i] + 1e-3).clamp(1.0, 95.0)

            f1_scaled = lo / self.sample_rate
            f2_scaled = hi / self.sample_rate

            w = (
                2 * f2_scaled * torch.sinc(2 * f2_scaled * n)
                - 2 * f1_scaled * torch.sinc(2 * f1_scaled * n)
            )
            filters.append(w.unsqueeze(0).unsqueeze(0))

        filters = torch.cat(filters, dim=0)
        x_reshaped = x.reshape(B * C, 1, T)
        out = F.conv1d(x_reshaped, filters, padding="same")

        return out.reshape(B, C, self.num_filters, T).permute(0, 2, 1, 3)


class DGNN(nn.Module):
    def __init__(self, num_filters=10, in_nodes=22, out_nodes=64):
        super().__init__()
        self.W_Q = nn.Linear(num_filters, num_filters)
        self.W_K = nn.Linear(num_filters, num_filters)
        self.W_V = nn.Linear(in_nodes, out_nodes)
        self.num_filters = num_filters

    def forward(self, x):
        B, num_bands, C, T = x.shape

        # Channel descriptors: B x C x F
        x_flat = x.mean(dim=-1).transpose(1, 2)

        Q = self.W_Q(x_flat)
        K = self.W_K(x_flat)

        A = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.num_filters)
        A = F.softmax(A, dim=-1)

        I = torch.eye(C, device=x.device, dtype=x.dtype).unsqueeze(0)
        A_hat = A + I

        degree = A_hat.sum(dim=-1).clamp_min(1e-6)
        D_hat_inv_sqrt = torch.diag_embed(1.0 / torch.sqrt(degree))
        norm_A = D_hat_inv_sqrt @ A_hat @ D_hat_inv_sqrt

        x_trans = x.permute(0, 1, 3, 2)   # B x F x T x C
        out = torch.einsum("bij,bntj->bnti", norm_A, x_trans)
        out = F.elu(self.W_V(out))

        return out.permute(0, 1, 3, 2)     # B x F x 64 x T


class SimplifiedBiMamba(nn.Module):
    """
    IMPORTANT:
    This class is named SimplifiedBiMamba in the supplied code, but
    the actual implementation is a bidirectional GRU.
    """
    def __init__(self, d_model=64):
        super().__init__()
        self.ssm = nn.GRU(
            input_size=d_model,
            hidden_size=d_model // 2,
            batch_first=True,
            bidirectional=True
        )

    def forward(self, x):
        out, _ = self.ssm(x.transpose(1, 2))
        return out.transpose(1, 2)


class SEAttention(nn.Module):
    def __init__(self, channel=64, reduction=16):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(channel, channel // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channel // reduction, channel, bias=False),
            nn.Sigmoid(),
        )

    def forward(self, x):
        b, c, _ = x.size()
        y = x.mean(dim=2)
        s = self.fc(y).view(b, c, 1)
        weighted_x = x * s.expand_as(x)
        return weighted_x.mean(dim=2)


class S3MambaDA(nn.Module):
    def __init__(self, num_classes=4, num_subjects=109):
        super().__init__()

        self.sinc_filter = SincFilterBank(
            in_channels=22,
            num_filters=10,
            kernel_size=81,
            sample_rate=250
        )

        self.dgnn = DGNN(
            num_filters=10,
            in_nodes=22,
            out_nodes=64
        )

        self.mamba = SimplifiedBiMamba(d_model=64)
        self.se_attention = SEAttention(channel=64)

        self.classifier = nn.Sequential(
            nn.BatchNorm1d(64),
            nn.Linear(64, num_classes)
        )

        self.domain_classifier = nn.Sequential(
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, num_subjects)
        )

        self.supcon_proj = nn.Sequential(
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, 128)
        )

    def forward(self, x, lambda_grl=1.0):
        f_out = self.sinc_filter(x)       # B F C T
        s_out = self.dgnn(f_out)          # B F D T

        pool_out = s_out.mean(dim=1)      # B D T
        t_out = self.mamba(pool_out)      # B D T
        z = self.se_attention(t_out)      # B D

        class_logits = self.classifier(z)

        z_grl = grl(z, lambda_grl)
        domain_logits = self.domain_classifier(z_grl)

        z_proj = F.normalize(self.supcon_proj(z), p=2, dim=1)

        return class_logits, domain_logits, z_proj


# -----------------------------
# 3. LOSSES
# -----------------------------
class SupConLoss(nn.Module):
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature

    def forward(self, features, labels):
        device = features.device
        batch_size = features.shape[0]

        sim = torch.matmul(features, features.T) / self.temperature

        labels = labels.contiguous().view(-1, 1)
        mask = torch.eq(labels, labels.T).float().to(device)

        logits_mask = torch.ones_like(mask)
        logits_mask.fill_diagonal_(0)
        mask = mask * logits_mask

        exp_logits = torch.exp(sim) * logits_mask
        log_prob = sim - torch.log(exp_logits.sum(1, keepdim=True) + 1e-6)

        positives = mask.sum(1)
        mean_log_prob_pos = (mask * log_prob).sum(1) / (positives + 1e-6)

        valid = positives > 0
        if valid.any():
            return -mean_log_prob_pos[valid].mean()
        return torch.zeros((), device=device, requires_grad=True)


def apply_adabn(model, target_dataloader, device, adaptation_trials=20):
    model.eval()

    target_samples = []
    trials_count = 0

    for x, _, _ in target_dataloader:
        target_samples.append(x)
        trials_count += x.size(0)
        if trials_count >= adaptation_trials:
            break

    if not target_samples:
        return model

    target_x = torch.cat(target_samples, dim=0)[:adaptation_trials].to(device)

    bn_modules = [
        m for m in model.modules()
        if isinstance(m, nn.modules.batchnorm._BatchNorm)
    ]

    if not bn_modules:
        return model

    saved = []
    for module in bn_modules:
        saved.append((module.training, module.momentum))
        module.reset_running_stats()
        module.momentum = 1.0
        module.train()

    with torch.no_grad():
        _ = model(target_x, lambda_grl=0.0)

    for module, (was_training, old_momentum) in zip(bn_modules, saved):
        module.momentum = 0.1
        module.eval()

    model.eval()
    return model


# -----------------------------
# 4. MODEL SMOKE TEST
# -----------------------------
def smoke_test():
    model = S3MambaDA(num_classes=N_CLASSES, num_subjects=DOMAIN_CLASSES).to(DEVICE)
    x = torch.randn(2, N_CHANNELS, int(FS * TMAX), device=DEVICE)

    with torch.no_grad():
        class_logits, domain_logits, z_proj = model(x, lambda_grl=0.0)

    print("Smoke test:")
    print("  input:", tuple(x.shape))
    print("  class logits:", tuple(class_logits.shape))
    print("  domain logits:", tuple(domain_logits.shape))
    print("  projection:", tuple(z_proj.shape))
    print("  params:", sum(p.numel() for p in model.parameters()))

smoke_test()


# -----------------------------
# 5. TRAINING + EVALUATION
# -----------------------------
def evaluate_large_scale_loso(
    data_dir=DATA_DIR,
    total_dataset_subjects=TOTAL_SUBJECTS,
    num_train_subjects=NUM_TRAIN_SUBJECTS,
    num_test_folds=NUM_TEST_FOLDS,
    num_epochs=NUM_EPOCHS,
    batch_size=BATCH_SIZE,
    seed=SEED,
):
    seed_everything(seed)

    available = discover_available_subjects(data_dir, total_dataset_subjects)
    if len(available) < num_test_folds:
        raise RuntimeError(
            f"Only {len(available)} subject folders found in {data_dir}; "
            f"need at least {num_test_folds}."
        )

    print(f"Available subject folders: {len(available)}")

    # Match supplied sampling idea, but restrict to folders that actually exist.
    rng = random.Random(seed)
    test_subjects = rng.sample(available, min(num_test_folds, len(available)))

    fold_rows = []
    pred_rows = []
    epoch_rows = []
    embedding_rows = []

    for fold_idx, test_subject in enumerate(test_subjects, start=1):
        print("=" * 80)
        print(f"FOLD {fold_idx}/{len(test_subjects)} | TEST SUBJECT S{test_subject:03d}")
        print("=" * 80)

        remaining = [s for s in available if s != test_subject]
        train_count = min(num_train_subjects, len(remaining))
        train_subjects = rng.sample(remaining, train_count)

        train_dataset = EEGMMIDB_Dataset(data_dir, subjects=train_subjects)
        test_dataset = EEGMMIDB_Dataset(data_dir, subjects=[test_subject])

        if len(train_dataset) == 0 or len(test_dataset) == 0:
            print("[WARN] Empty fold; skipping.")
            continue

        train_loader = DataLoader(
            train_dataset,
            batch_size=batch_size,
            shuffle=True,
            num_workers=0,
            pin_memory=False
        )

        test_loader = DataLoader(
            test_dataset,
            batch_size=batch_size,
            shuffle=False,
            num_workers=0,
            pin_memory=False
        )

        model = S3MambaDA(
            num_classes=N_CLASSES,
            num_subjects=total_dataset_subjects
        ).to(DEVICE)

        criterion_cls = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
        criterion_domain = nn.CrossEntropyLoss()
        criterion_supcon = SupConLoss(temperature=SUPCON_TEMP)

        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=LR,
            weight_decay=WEIGHT_DECAY
        )

        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=num_epochs,
            eta_min=1e-5
        )

        use_amp = DEVICE.type == "cuda"
        scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

        for epoch in range(num_epochs):
            model.train()

            epoch_loss = 0.0
            epoch_cls = 0.0
            epoch_domain = 0.0
            epoch_supcon = 0.0
            seen = 0

            total_batches = len(train_loader)

            for batch_idx, (x, y, s) in enumerate(train_loader):
                x = x.to(DEVICE)
                y = y.to(DEVICE)
                s = s.to(DEVICE)

                p = float(batch_idx + epoch * total_batches) / max(
                    1, num_epochs * total_batches
                )
                lambda_grl = 2.0 / (1.0 + np.exp(-10.0 * p)) - 1.0

                optimizer.zero_grad(set_to_none=True)

                with torch.autocast(
                    device_type=DEVICE.type,
                    enabled=use_amp
                ):
                    class_logits, domain_logits, z_proj = model(
                        x, lambda_grl=lambda_grl
                    )

                    loss_cls = criterion_cls(class_logits, y)
                    loss_domain = criterion_domain(domain_logits, s)
                    loss_supcon = criterion_supcon(z_proj, y)

                    loss_total = (
                        loss_cls
                        + DOMAIN_WEIGHT * loss_domain
                        + SUPCON_WEIGHT * loss_supcon
                    )

                scaler.scale(loss_total).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    max_norm=GRAD_CLIP
                )
                scaler.step(optimizer)
                scaler.update()

                bs = x.size(0)
                seen += bs
                epoch_loss += float(loss_total.detach().cpu()) * bs
                epoch_cls += float(loss_cls.detach().cpu()) * bs
                epoch_domain += float(loss_domain.detach().cpu()) * bs
                epoch_supcon += float(loss_supcon.detach().cpu()) * bs

            scheduler.step()

            epoch_rows.append({
                "fold": fold_idx,
                "test_subject": test_subject,
                "epoch": epoch + 1,
                "loss_total": epoch_loss / max(1, seen),
                "loss_cls": epoch_cls / max(1, seen),
                "loss_domain": epoch_domain / max(1, seen),
                "loss_supcon": epoch_supcon / max(1, seen),
                "lr": optimizer.param_groups[0]["lr"],
            })

            if (epoch + 1) % 10 == 0 or epoch == 0:
                print(
                    f"Epoch {epoch+1:03d}/{num_epochs} | "
                    f"Loss {epoch_loss/max(1,seen):.4f}"
                )

        # Unsupervised target-statistics adaptation.
        model = apply_adabn(
            model,
            test_loader,
            DEVICE,
            adaptation_trials=len(test_dataset)
        )

        model.eval()

        all_preds, all_labels, all_probs, all_embeds, all_subjects = [], [], [], [], []

        with torch.no_grad():
            for x, y, s in test_loader:
                x = x.to(DEVICE)
                y = y.to(DEVICE)

                class_logits, _, z_proj = model(x, lambda_grl=0.0)

                probs = torch.softmax(class_logits, dim=1)
                preds = torch.argmax(probs, dim=1)

                all_preds.extend(preds.cpu().numpy().tolist())
                all_labels.extend(y.cpu().numpy().tolist())
                all_probs.append(probs.cpu().numpy())
                all_embeds.append(z_proj.cpu().numpy())
                all_subjects.extend(s.cpu().numpy().tolist())

        all_probs = np.concatenate(all_probs, axis=0)
        all_embeds = np.concatenate(all_embeds, axis=0)

        acc = accuracy_score(all_labels, all_preds)
        kappa = cohen_kappa_score(all_labels, all_preds)

        report = classification_report(
            all_labels,
            all_preds,
            labels=list(range(N_CLASSES)),
            target_names=CLASS_NAMES,
            output_dict=True,
            zero_division=0
        )

        fold_rows.append({
            "fold": fold_idx,
            "test_subject": test_subject,
            "n_train_trials": len(train_dataset),
            "n_test_trials": len(test_dataset),
            "accuracy": acc,
            "kappa": kappa,
            "precision_macro": report["macro avg"]["precision"],
            "recall_macro": report["macro avg"]["recall"],
            "f1_macro": report["macro avg"]["f1-score"],
        })

        for i in range(len(all_labels)):
            pred_rows.append({
                "fold": fold_idx,
                "test_subject": test_subject,
                "true_label": int(all_labels[i]),
                "pred_label": int(all_preds[i]),
                **{f"prob_{c}": float(all_probs[i, c]) for c in range(N_CLASSES)}
            })

            embedding_rows.append({
                "fold": fold_idx,
                "test_subject": test_subject,
                "true_label": int(all_labels[i]),
                **{
                    f"z_{j}": float(all_embeds[i, j])
                    for j in range(all_embeds.shape[1])
                }
            })

        print(
            f"Subject S{test_subject:03d} | "
            f"Accuracy={acc*100:.2f}% | Kappa={kappa:.4f}"
        )

    # Save all experiment artifacts.
    fold_df = pd.DataFrame(fold_rows)
    pred_df = pd.DataFrame(pred_rows)
    epoch_df = pd.DataFrame(epoch_rows)
    emb_df = pd.DataFrame(embedding_rows)

    fold_df.to_csv(RESULTS_DIR / "fold_metrics.csv", index=False)
    pred_df.to_csv(RESULTS_DIR / "test_predictions.csv", index=False)
    epoch_df.to_csv(RESULTS_DIR / "epoch_history.csv", index=False)
    emb_df.to_csv(RESULTS_DIR / "test_embeddings.csv", index=False)

    summary = {
        "mean_accuracy": float(fold_df["accuracy"].mean()) if len(fold_df) else None,
        "std_accuracy": float(fold_df["accuracy"].std(ddof=0)) if len(fold_df) else None,
        "mean_kappa": float(fold_df["kappa"].mean()) if len(fold_df) else None,
        "std_kappa": float(fold_df["kappa"].std(ddof=0)) if len(fold_df) else None,
        "num_folds_completed": int(len(fold_df)),
        "test_subjects": test_subjects,
        "device": str(DEVICE),
    }

    with open(RESULTS_DIR / "summary.json", "w") as f:
        json.dump(summary, f, indent=2)

    print("\nFINAL SUMMARY")
    print(json.dumps(summary, indent=2))

    return fold_df, pred_df, epoch_df, emb_df


# ------------------------------------------------------------
# 6. PAPER-READY FIGURES / INFOGRAPHICS
# ------------------------------------------------------------
def plot_architecture():
    fig, ax = plt.subplots(figsize=(15, 7))
    ax.set_xlim(0, 15)
    ax.set_ylim(0, 8)
    ax.axis("off")

    blocks = [
        (0.3, 5.5, 1.7, 1.0, "Raw EEG\n(B,22,1000)"),
        (2.4, 5.5, 1.9, 1.0, "Per-trial\nZ-score"),
        (4.7, 5.5, 2.0, 1.0, "Learnable\nSinc Bank\n10 bands"),
        (7.1, 5.5, 2.0, 1.0, "Dynamic GNN\nAdaptive graph\n22 → 64"),
        (9.5, 5.5, 2.0, 1.0, "BiGRU\nTemporal\nmodeling"),
        (11.9, 5.5, 2.2, 1.0, "SE Attention\n+ mean pool\nz ∈ R64"),
    ]

    for x, y, w, h, txt in blocks:
        r = plt.Rectangle((x, y), w, h, fill=False, linewidth=1.8)
        ax.add_patch(r)
        ax.text(x+w/2, y+h/2, txt, ha="center", va="center", fontsize=11)

    for i in range(len(blocks)-1):
        x1 = blocks[i][0] + blocks[i][2]
        x2 = blocks[i+1][0]
        y = blocks[i][1] + blocks[i][3]/2
        ax.annotate("", xy=(x2, y), xytext=(x1, y),
                    arrowprops=dict(arrowstyle="->", linewidth=1.5))

    # Branches
    ax.plot([13.0, 13.0], [5.5, 3.8], linewidth=1.5)
    ax.annotate("", xy=(10.7, 3.2), xytext=(13.0, 3.8),
                arrowprops=dict(arrowstyle="->", linewidth=1.5))
    ax.annotate("", xy=(13.0, 1.8), xytext=(13.0, 3.8),
                arrowprops=dict(arrowstyle="->", linewidth=1.5))
    ax.annotate("", xy=(6.8, 1.8), xytext=(13.0, 3.8),
                arrowprops=dict(arrowstyle="->", linewidth=1.5))

    outputs = [
        (5.4, 0.7, 2.8, 1.0, "Class Head\n4 MI classes"),
        (9.1, 0.7, 3.2, 1.0, "GRL + Domain Head\nSubject alignment"),
        (3.2, 2.6, 3.6, 1.0, "Projection Head\n128-D SupCon embedding"),
    ]

    for x, y, w, h, txt in outputs:
        r = plt.Rectangle((x, y), w, h, fill=False, linewidth=1.6)
        ax.add_patch(r)
        ax.text(x+w/2, y+h/2, txt, ha="center", va="center", fontsize=10)

    ax.text(7.5, 7.6,
            "S³ Spectral-Spatial Domain-Adaptive EEG Classification Pipeline",
            ha="center", va="center", fontsize=16, fontweight="bold")
    ax.text(7.5, 7.1,
            "Code-aligned architecture: learnable frequency decomposition + dynamic topology + bidirectional GRU + GRL + SupCon + AdaBN",
            ha="center", va="center", fontsize=10)

    fig.tight_layout()
    fig.savefig(FIG_DIR / "Fig1_Architecture.png", dpi=400, bbox_inches="tight")
    plt.close(fig)


def plot_training_curves(epoch_df):
    if epoch_df.empty:
        return

    # Aggregate across folds by epoch.
    g = epoch_df.groupby("epoch").agg({
        "loss_total": "mean",
        "loss_cls": "mean",
        "loss_domain": "mean",
        "loss_supcon": "mean",
    }).reset_index()

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(g["epoch"], g["loss_total"], label="Total loss", linewidth=2)
    ax.plot(g["epoch"], g["loss_cls"], label="Classification loss", linewidth=1.5)
    ax.plot(g["epoch"], g["loss_domain"], label="Domain loss", linewidth=1.5)
    ax.plot(g["epoch"], g["loss_supcon"], label="SupCon loss", linewidth=1.5)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.set_title("Training Loss Components")
    ax.grid(True, alpha=0.25)
    ax.legend()
    fig.tight_layout()
    fig.savefig(FIG_DIR / "Fig2_TrainingLoss.png", dpi=400, bbox_inches="tight")
    plt.close(fig)


def plot_subject_accuracy(fold_df):
    if fold_df.empty:
        return

    fig, ax = plt.subplots(figsize=(9, 5))
    labels = [f"S{s:03d}" for s in fold_df["test_subject"]]
    ax.bar(labels, fold_df["accuracy"] * 100)
    ax.axhline(25, linestyle="--", linewidth=1.3, label="4-class chance")
    ax.set_ylabel("Accuracy (%)")
    ax.set_xlabel("Held-out subject")
    ax.set_title("Subject-Independent Test Accuracy by Fold")
    ax.legend()
    ax.grid(axis="y", alpha=0.25)
    fig.tight_layout()
    fig.savefig(FIG_DIR / "Fig3_FoldAccuracy.png", dpi=400, bbox_inches="tight")
    plt.close(fig)


def plot_confusion(pred_df):
    if pred_df.empty:
        return

    cm = confusion_matrix(
        pred_df["true_label"],
        pred_df["pred_label"],
        labels=list(range(N_CLASSES))
    )

    cm_norm = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)

    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(cm_norm, interpolation="nearest")
    ax.set_title("Normalized Confusion Matrix")
    ax.set_xlabel("Predicted class")
    ax.set_ylabel("True class")
    ax.set_xticks(range(N_CLASSES), CLASS_NAMES, rotation=25, ha="right")
    ax.set_yticks(range(N_CLASSES), CLASS_NAMES)

    for i in range(N_CLASSES):
        for j in range(N_CLASSES):
            ax.text(j, i, f"{cm_norm[i, j]*100:.1f}%",
                    ha="center", va="center")

    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    fig.savefig(FIG_DIR / "Fig4_ConfusionMatrix.png", dpi=400, bbox_inches="tight")
    plt.close(fig)


def plot_class_metrics(pred_df):
    if pred_df.empty:
        return

    report = classification_report(
        pred_df["true_label"],
        pred_df["pred_label"],
        labels=list(range(N_CLASSES)),
        target_names=CLASS_NAMES,
        output_dict=True,
        zero_division=0
    )

    precision = [report[c]["precision"] * 100 for c in CLASS_NAMES]
    recall = [report[c]["recall"] * 100 for c in CLASS_NAMES]
    f1 = [report[c]["f1-score"] * 100 for c in CLASS_NAMES]

    x = np.arange(N_CLASSES)
    width = 0.25

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.bar(x-width, precision, width, label="Precision")
    ax.bar(x, recall, width, label="Recall")
    ax.bar(x+width, f1, width, label="F1")
    ax.set_xticks(x, CLASS_NAMES, rotation=20, ha="right")
    ax.set_ylim(0, 100)
    ax.set_ylabel("Score (%)")
    ax.set_title("Per-Class Classification Metrics")
    ax.legend()
    ax.grid(axis="y", alpha=0.25)
    fig.tight_layout()
    fig.savefig(FIG_DIR / "Fig5_ClassMetrics.png", dpi=400, bbox_inches="tight")
    plt.close(fig)


def plot_roc(pred_df):
    if pred_df.empty:
        return

    y_true = pred_df["true_label"].to_numpy()
    y_prob = pred_df[[f"prob_{c}" for c in range(N_CLASSES)]].to_numpy()

    fig, ax = plt.subplots(figsize=(7, 6))

    for c in range(N_CLASSES):
        binary = (y_true == c).astype(int)
        if binary.min() == binary.max():
            continue
        fpr, tpr, _ = roc_curve(binary, y_prob[:, c])
        roc_auc = auc(fpr, tpr)
        ax.plot(fpr, tpr, linewidth=2,
                label=f"{CLASS_NAMES[c]} (AUC={roc_auc:.3f})")

    ax.plot([0, 1], [0, 1], linestyle="--", linewidth=1)
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title("One-vs-Rest ROC Curves")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.25)
    fig.tight_layout()
    fig.savefig(FIG_DIR / "Fig6_ROC_AUC.png", dpi=400, bbox_inches="tight")
    plt.close(fig)


def plot_tsne(emb_df, seed=SEED):
    if emb_df.empty:
        return

    z_cols = [c for c in emb_df.columns if c.startswith("z_")]
    if len(emb_df) < 10 or len(z_cols) < 2:
        return

    X = emb_df[z_cols].to_numpy()
    y = emb_df["true_label"].to_numpy()

    perplexity = min(30, max(5, (len(X) - 1) // 3))

    tsne = TSNE(
        n_components=2,
        perplexity=perplexity,
        init="pca",
        learning_rate="auto",
        random_state=seed
    )

    Z = tsne.fit_transform(X)

    fig, ax = plt.subplots(figsize=(8, 6))
    for c in range(N_CLASSES):
        mask = y == c
        ax.scatter(Z[mask, 0], Z[mask, 1], s=12, label=CLASS_NAMES[c], alpha=0.75)

    ax.set_title("t-SNE Projection of Test-Time Class Embeddings")
    ax.set_xlabel("t-SNE dimension 1")
    ax.set_ylabel("t-SNE dimension 2")
    ax.legend(markerscale=1.5, fontsize=8)
    ax.grid(True, alpha=0.15)
    fig.tight_layout()
    fig.savefig(FIG_DIR / "Fig7_tSNE.png", dpi=400, bbox_inches="tight")
    plt.close(fig)


def generate_all_figures():
    plot_architecture()

    required = [
        "fold_metrics.csv",
        "test_predictions.csv",
        "epoch_history.csv",
        "test_embeddings.csv"
    ]

    if not all((RESULTS_DIR / f).exists() for f in required):
        print("Only architecture figure was generated. Run the experiment first.")
        return

    fold_df = pd.read_csv(RESULTS_DIR / "fold_metrics.csv")
    pred_df = pd.read_csv(RESULTS_DIR / "test_predictions.csv")
    epoch_df = pd.read_csv(RESULTS_DIR / "epoch_history.csv")
    emb_df = pd.read_csv(RESULTS_DIR / "test_embeddings.csv")

    plot_training_curves(epoch_df)
    plot_subject_accuracy(fold_df)
    plot_confusion(pred_df)
    plot_class_metrics(pred_df)
    plot_roc(pred_df)
    plot_tsne(emb_df)

    print("Figures written to:", FIG_DIR.resolve())
    for p in sorted(FIG_DIR.glob("*.png")):
        print(" -", p.name)


# ------------------------------------------------------------
# 7. RUN
# ------------------------------------------------------------
# For a first smoke test, keep NUM_EPOCHS small (e.g., 2) by changing
# the configuration above, then restore the final value for the paper.
#
# Example:
# fold_df, pred_df, epoch_df, emb_df = evaluate_large_scale_loso()
# generate_all_figures()

# Architecture is always safe to generate immediately.
generate_all_figures()


# ------------------------------------------------------------
# 8. OPTIONAL: PAPER RESULTS TEXT EXPORT
# ------------------------------------------------------------
def export_results_text():
    fold_file = RESULTS_DIR / "fold_metrics.csv"
    if not fold_file.exists():
        print("Run the experiment first.")
        return

    fold_df = pd.read_csv(fold_file)
    if fold_df.empty:
        print("No completed folds.")
        return

    acc_mean = fold_df["accuracy"].mean() * 100
    acc_std = fold_df["accuracy"].std(ddof=0) * 100
    kap_mean = fold_df["kappa"].mean()
    kap_std = fold_df["kappa"].std(ddof=0)

    text = f"""
RESULTS SUMMARY FOR PAPER
=========================
Completed folds: {len(fold_df)}
Subjects: {", ".join("S%03d" % s for s in fold_df["test_subject"])}

Mean Accuracy: {acc_mean:.2f}% ± {acc_std:.2f}%
Mean Cohen's Kappa: {kap_mean:.4f} ± {kap_std:.4f}%

Per-fold:
{fold_df.to_string(index=False)}
""".strip()

    (RESULTS_DIR / "paper_results.txt").write_text(text)
    print(text)

# export_results_text()
